# Chapter IV — Side-Loaded Plate (VFM)

This notebook migrates the legacy `VFM_inverse` workflow into the chapter structure with three parts:

1. **Noise-study statistics** (aligned with `02_noise_study`)
2. **Subregion study** (aligned with `04_missing_data`)
3. **Padding investigation**

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
from scipy.spatial import cKDTree

from phd.io import get_side_loaded_plate_dataset_path, load_side_loaded_plate_reference_raw
from phd.plot import get_current_config as plt_cfg, book_config

book_config.set_as_current()
page_width = plt_cfg().page_width

save_fig = False
save_table = False
if save_fig:
    mpl.rcParams["pgf.texsystem"] = "pdflatex"


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for parent in [current, *current.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("Could not find project root (missing pyproject.toml).")


PROJECT_ROOT = find_project_root(Path.cwd())
CHAPTER_DIR = PROJECT_ROOT / "chapters" / "IV_MaterialCharacterization" / "01_side_loaded_plate"
IMAGE_DIR = CHAPTER_DIR / "images"
PDF_DIR = IMAGE_DIR / "pdf"
TABLE_DIR = CHAPTER_DIR / "tables"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# DIC datasets resolved via IO dataset utility
DIC_DATA_ROOT = get_side_loaded_plate_dataset_path("dic")
NOISE_SUMMARY_PATH = CHAPTER_DIR / "tables" / "02_noise_study_stats.json"
MISSING_DATA_SUMMARY_PATH = CHAPTER_DIR / "tables" / "VFM_missing_data.json"
    
print(f"Project root: {PROJECT_ROOT}")
print(f"Chapter dir: {CHAPTER_DIR}")
print(f"DIC data root: {DIC_DATA_ROOT}")
print(f"Noise-study summary: {NOISE_SUMMARY_PATH}")
print(f"Missing-data summary: {MISSING_DATA_SUMMARY_PATH}")

Using backend: jax
Other supported backends: tensorflow.compat.v1, tensorflow, pytorch, paddle.
paddle supports more examples now and is recommended.
Enable just-in-time compilation with XLA.



Project root: /home/r2d2/code/PhD
Chapter dir: /home/r2d2/code/PhD/chapters/IV_MaterialCharacterization/01_side_loaded_plate
DIC data root: /home/r2d2/code/PhD/src/phd/io/dataset/side_loaded_plate/dic
Noise-study summary: /home/r2d2/code/PhD/chapters/IV_MaterialCharacterization/01_side_loaded_plate/tables/02_noise_study_stats.json
Missing-data summary: /home/r2d2/code/PhD/chapters/IV_MaterialCharacterization/01_side_loaded_plate/tables/VFM_missing_data.json


In [2]:
# Load FEM data and build interpolation functions (legacy VFM setup)
FEM_dataset = "100x100mm.dat"
L = 100.0  # mm
m = 0.3    # N/mm
b = 50.0   # N
F = m * L**2 / 2 + b * L

fem_path = get_side_loaded_plate_dataset_path(FEM_dataset)
raw = load_side_loaded_plate_reference_raw(FEM_dataset)

X_val = raw[:, :2]
u_val = raw[:, 2:4]
strain_val = raw[:, 4:7]
stress_val = raw[:, 7:10]
solution_val = np.hstack((u_val, stress_val))

n_mesh_points = int(np.sqrt(X_val.shape[0]))
x_grid = np.linspace(0, L, n_mesh_points)
y_grid = np.linspace(0, L, n_mesh_points)


def create_interpolation_fn(data_array):
    num_components = data_array.shape[1]
    interpolators = []
    for i in range(num_components):
        interp = RegularGridInterpolator(
            (x_grid, y_grid),
            data_array[:, i].reshape(n_mesh_points, n_mesh_points).T,
        )
        interpolators.append(interp)

    def interpolation_fn(x):
        return np.array([interp((x[:, 0], x[:, 1])) for interp in interpolators]).T

    return interpolation_fn


solution_fn = create_interpolation_fn(solution_val)
strain_fn = create_interpolation_fn(strain_val)
print(f"FEM dataset: {fem_path}")
print(f"Mesh points: {n_mesh_points} x {n_mesh_points}")

FEM dataset: /home/r2d2/code/PhD/src/phd/io/dataset/side_loaded_plate/100x100mm.dat
Mesh points: 100 x 100


In [3]:
def postprocess_dic_data(
    Xg,
    Yg,
    Exx,
    Eyy,
    Exy,
    DIC_region,
    num_measurments,
    pad_extrapolate=False,
    verbose=True,
):
    m_pts, n_pts = Xg.shape
    i_full = np.round(np.linspace(0, m_pts - 1, min(m_pts, num_measurments))).astype(int)
    j_full = np.round(np.linspace(0, n_pts - 1, min(n_pts, num_measurments))).astype(int)
    n_full = i_full.size

    sel_full = np.ix_(i_full, j_full)
    Xf = Xg[sel_full]
    Yf = Yg[sel_full]
    Exx_f = Exx[sel_full]
    Eyy_f = Eyy[sel_full]
    Exy_f = Exy[sel_full]

    xmin, ymin, xmax, ymax = DIC_region
    mask = (Xf >= xmin) & (Xf <= xmax) & (Yf >= ymin) & (Yf <= ymax)
    rows_r, cols_r = np.unique(np.where(mask)[0]), np.unique(np.where(mask)[1])

    if verbose:
        if pad_extrapolate:
            print(
                f"DIC grid: {m_pts} x {n_pts} --> Subsampled grid (including padding): {n_full} x {n_full} --> Region grid: {rows_r.size} x {cols_r.size}"
            )
        else:
            print(
                f"DIC grid: {m_pts} x {n_pts} --> Subsampled grid: {n_full} x {n_full} --> Region grid (points used): {rows_r.size} x {cols_r.size}"
            )

    if pad_extrapolate:
        pts_in = np.column_stack([Xf[mask].ravel(), Yf[mask].ravel()])
        vals_in = np.column_stack([Exx_f[mask].ravel(), Eyy_f[mask].ravel(), Exy_f[mask].ravel()])
        tree = cKDTree(pts_in)
        pts_all = np.column_stack([Xf.ravel(), Yf.ravel()])
        _, idx = tree.query(pts_all)
        coords = pts_all
        data = vals_in[idx]
    else:
        coords = np.column_stack([Xf[mask].ravel(), Yf[mask].ravel()])
        data = np.column_stack([Exx_f[mask].ravel(), Eyy_f[mask].ravel(), Exy_f[mask].ravel()])

    return coords, data


def load_data(
    DIC_dataset_path,
    DIC_dataset_number,
    strain_fn,
    num_measurments,
    noise_magnitude,
    DIC_region,
    pad_extrapolate=False,
    verbose=True,
):
    if DIC_dataset_path != "no_dataset":
        base = Path(DIC_dataset_path)
        n_idx = DIC_dataset_number
        Xg = pd.read_csv(base / "x" / f"x_{n_idx}.csv", delimiter=";").dropna(axis=1).to_numpy()
        Yg = pd.read_csv(base / "y" / f"y_{n_idx}.csv", delimiter=";").dropna(axis=1).to_numpy()
        Exx = pd.read_csv(base / "exx" / f"exx_{n_idx}.csv", delimiter=";").dropna(axis=1).to_numpy()
        Eyy = pd.read_csv(base / "eyy" / f"eyy_{n_idx}.csv", delimiter=";").dropna(axis=1).to_numpy()
        Exy = pd.read_csv(base / "exy" / f"exy_{n_idx}.csv", delimiter=";").dropna(axis=1).to_numpy()

        coords, data = postprocess_dic_data(
            Xg,
            Yg,
            Exx,
            Eyy,
            Exy,
            DIC_region=DIC_region,
            num_measurments=num_measurments,
            pad_extrapolate=pad_extrapolate,
            verbose=verbose,
        )
    else:
        xmin, ymin, xmax, ymax = DIC_region
        xs = np.linspace(xmin, xmax, num_measurments)
        ys = np.linspace(ymin, ymax, num_measurments)
        Xg, Yg = np.meshgrid(xs, ys)
        coords = np.column_stack([Xg.ravel(), Yg.ravel()])
        data = strain_fn(coords)
        data += np.random.normal(0, noise_magnitude, data.shape)
        if verbose:
            print(f"Dic grid: {num_measurments} x {num_measurments} with {noise_magnitude} magnitude noise")

    return coords, data


def calc_parameters(Eps1, Eps2, Eps6):
    A = np.zeros((2, 2))
    B = np.zeros(2)

    A[0, 0] = np.mean(Eps2) * L**2
    A[0, 1] = np.mean(Eps1) * L**2
    B[0] = 0

    A[1, 0] = np.mean(Eps1) * L**2
    A[1, 1] = np.mean(Eps2) * L**2
    B[1] = F * L

    Q11, Q12 = np.linalg.solve(A, B)

    mu_id = (Q11 - Q12) / 2
    lambda_id = Q12
    nu_id = lambda_id / (2 * (mu_id + lambda_id))
    E_id = mu_id * (3 * lambda_id + 2 * mu_id) / (lambda_id + mu_id)
    return E_id, nu_id


def run_vfm_trials(
    DIC_dataset_path,
    num_measurments,
    DIC_region,
    noise_magnitude=1e-6,
    n_trials=10,
    pad_extrapolate=False,
):
    E_list = []
    Nu_list = []

    for idx in range(1, n_trials + 1):
        dic_number = idx if DIC_dataset_path != "no_dataset" else 0
        _, DIC_data = load_data(
            DIC_dataset_path=DIC_dataset_path,
            DIC_dataset_number=dic_number,
            strain_fn=strain_fn,
            num_measurments=num_measurments,
            noise_magnitude=noise_magnitude,
            DIC_region=DIC_region,
            pad_extrapolate=pad_extrapolate,
            verbose=(idx == 1),
        )

        Eps1 = DIC_data[:, 0].reshape(-1, 1)
        Eps2 = DIC_data[:, 1].reshape(-1, 1)
        Eps6 = DIC_data[:, 2].reshape(-1, 1)

        E_id, Nu_id = calc_parameters(Eps1, Eps2, Eps6)
        E_list.append(E_id)
        Nu_list.append(Nu_id)

    return np.array(E_list), np.array(Nu_list)

## Part 1 — Noise-Study Statistics (aligned with `02_noise_study`)

Computes VFM statistics over DIC datasets for each camera resolution and optionally updates the `02_noise_study` summary JSON.

In [4]:
resolutions = ["5MP", "2MP", "0.4MP"]
prop = 0.0
num_measurments = 1000
E_actual = 210.0  # GPa
Nu_actual = 0.3
update_summary_json = True

DIC_region = [prop * L, prop * L, (1 - prop) * L, (1 - prop) * L]  # [xmin, ymin, xmax, ymax]
summary_updates = {}

for camera_resolution in resolutions:
    dic_path = DIC_DATA_ROOT / camera_resolution / "1noise"
    E_list, Nu_list = run_vfm_trials(
        DIC_dataset_path=dic_path,
        num_measurments=num_measurments,
        DIC_region=DIC_region,
        noise_magnitude=1e-6,
        n_trials=10,
        pad_extrapolate=False,
    )

    E_gpa = E_list / 1e3
    nu_vals = Nu_list

    summary_updates[camera_resolution] = {
        "VFM": {
            "E": {
                "mean": float(np.mean(E_gpa)),
                "std": float(np.std(E_gpa)),
                "mean_rel_error": float(np.mean(np.abs((E_gpa - E_actual) / E_actual)) * 100),
                "std_rel_error": float(np.std(np.abs((E_gpa - E_actual) / E_actual)) * 100),
            },
            "nu": {
                "mean": float(np.mean(nu_vals)),
                "std": float(np.std(nu_vals)),
                "mean_rel_error": float(np.mean(np.abs((nu_vals - Nu_actual) / Nu_actual)) * 100),
                "std_rel_error": float(np.std(np.abs((nu_vals - Nu_actual) / Nu_actual)) * 100),
            },
        }
    }

    print(
        f"{camera_resolution}: E = {np.mean(E_gpa):.4f} ± {np.std(E_gpa):.4f} GPa, "
        f"nu = {np.mean(nu_vals):.4f} ± {np.std(nu_vals):.4f}"
    )

if update_summary_json:
    summary_data = {}
    if NOISE_SUMMARY_PATH.exists():
        with open(NOISE_SUMMARY_PATH, "r") as json_file:
            summary_data = json.load(json_file)

    for camera_resolution in resolutions:
        summary_data.setdefault(camera_resolution, {}).update(summary_updates[camera_resolution])

    with open(NOISE_SUMMARY_PATH, "w") as json_file:
        json.dump(summary_data, json_file, indent=4)

    print(f"Saved: {NOISE_SUMMARY_PATH}")

DIC grid: 191 x 192 --> Subsampled grid: 191 x 191 --> Region grid (points used): 191 x 192
5MP: E = 209.8776 ± 0.0400 GPa, nu = 0.3003 ± 0.0001
DIC grid: 114 x 115 --> Subsampled grid: 114 x 114 --> Region grid (points used): 114 x 115
2MP: E = 210.2148 ± 0.0885 GPa, nu = 0.2989 ± 0.0002
DIC grid: 39 x 41 --> Subsampled grid: 39 x 39 --> Region grid (points used): 39 x 41
0.4MP: E = 209.8594 ± 0.3159 GPa, nu = 0.3004 ± 0.0007
Updated: /home/r2d2/code/PhD/chapters/IV_MaterialCharacterization/01_side_loaded_plate/tables/02_noise_study_stats.json


## Part 2 — Subregion Study (aligned with `04_missing_data`)

Uses only a subregion of the plate to estimate material parameters with synthetic strain data, matching the missing-data style study intent.

In [4]:
prop = [0.1, 0.1, 0.6, 0.6]
DIC_region = [prop[0] * L, prop[1] * L, prop[2] * L, prop[3] * L]  # [xmin, ymin, xmax, ymax]
n_measurments = 40

E_list, Nu_list = run_vfm_trials(
    DIC_dataset_path="no_dataset",
    num_measurments=n_measurments,
    DIC_region=DIC_region,
    noise_magnitude=1e-4,
    n_trials=10,
    pad_extrapolate=False,
)

E_mean_gpa = float(np.mean(E_list) / 1e3)
E_std_gpa = float(np.std(E_list) / 1e3)
nu_mean = float(np.mean(Nu_list))
nu_std = float(np.std(Nu_list))

print(
    f"Identified: E = {E_mean_gpa:.4f} ± {E_std_gpa:.4f} GPa, "
    f"ν = {nu_mean:.4f} ± {nu_std:.4f}"
)

missing_data_summary = {
    "vfm_missing_data": {
        "settings": {
            "dic_region": [float(v) for v in DIC_region],
            "num_measurements": int(n_measurments),
            "noise_magnitude": 1e-4,
            "n_trials": 10,
            "pad_extrapolate": False,
        },
        "E_GPa": {
            "mean": E_mean_gpa,
            "std": E_std_gpa,
            "samples": (E_list / 1e3).tolist(),
        },
        "nu": {
            "mean": nu_mean,
            "std": nu_std,
            "samples": Nu_list.tolist(),
        },
    }
}

with open(MISSING_DATA_SUMMARY_PATH, "w") as json_file:
    json.dump(missing_data_summary, json_file, indent=4)

print(f"Saved: {MISSING_DATA_SUMMARY_PATH}")

Dic grid: 40 x 40 with 0.0001 magnitude noise
Identified: E = 226.8392 ± 1.8519 GPa, ν = 0.2870 ± 0.0032
Saved: /home/r2d2/code/PhD/chapters/IV_MaterialCharacterization/01_side_loaded_plate/tables/VFM_missing_data.json


## Extra — Padding Investigation

Compare VFM identification with and without padding/extrapolation over a subregion using DIC datasets.

In [5]:
camera_resolution = "2MP"
DIC_dataset_path = DIC_DATA_ROOT / camera_resolution / "1noise"
prop = 0.1
DIC_region = [prop * L, prop * L, (1 - prop) * L, (1 - prop) * L]  # [xmin, ymin, xmax, ymax]
num_measurments = 6

for pad_extrapolate in [True, False]:
    E_list, Nu_list = run_vfm_trials(
        DIC_dataset_path=DIC_dataset_path,
        num_measurments=num_measurments,
        DIC_region=DIC_region,
        noise_magnitude=1e-6,
        n_trials=10,
        pad_extrapolate=pad_extrapolate,
    )

    label = "Padding" if pad_extrapolate else "No padding"
    print(
        f"{label}: E = {np.mean(E_list) / 1e3:.4f} ± {np.std(E_list) / 1e3:.4f} GPa, "
        f"ν = {np.mean(Nu_list):.4f} ± {np.std(Nu_list):.4f}"
    )

DIC grid: 114 x 115 --> Subsampled grid (including padding): 6 x 6 --> Region grid: 4 x 4
Padding: E = 211.3346 ± 0.1827 GPa, ν = 0.2978 ± 0.0004
DIC grid: 114 x 115 --> Subsampled grid: 6 x 6 --> Region grid (points used): 4 x 4
No padding: E = 211.7390 ± 0.1643 GPa, ν = 0.2971 ± 0.0003
